# 03 — Evaluation & Model Comparison

This notebook evaluates **five models** on the same test set and provides a side-by-side comparison.

**Models:**
1. **Base** — `Qwen2.5-Math-7B` (no fine-tuning)
2. **Instruct** — `Qwen2.5-Math-7B-Instruct` (no fine-tuning)
3. **SFT** — Base + SFT LoRA adapter (NuminaMath-CoT)
4. **GRPO-from-SFT** — Merged SFT model + GRPO LoRA adapter
5. **GRPO-from-Instruct** — Instruct model + GRPO LoRA adapter

**Eval dataset:** In-distribution val split from NuminaMath-CoT (same sources as SFT training, held-out examples). Toggle `USE_VAL_SPLIT = False` to use the official test split instead.

**Metrics:** Exact-match accuracy on `\boxed{}` answers with symbolic equality fallback.

## 1. Setup

In [1]:
!git clone https://github.com/Roogard/math-rl-tuning.git
%cd math-rl-tuning

# Install the package
!pip install -e ".[viz]" --quiet
!pip install bitsandbytes latex2sympy2 --quiet
!pip install math-verify

Cloning into 'math-rl-tuning'...
remote: Enumerating objects: 511, done.
remote: Counting objects: 100% (159/159), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 511 (delta 88), reused 102 (delta 44), pack-reused 352 (from 1)
Receiving objects: 100% (511/511), 4.45 MiB | 17.48 MiB/s, done.
Resolving deltas: 100% (312/312), done.
/content/math-rl-tuning
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.8/89.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 48.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runt

In [2]:
from math_rl_tuning.config import load_config
from math_rl_tuning.utils import setup_hf_token, mount_google_drive

cfg = load_config()
setup_hf_token()
mount_google_drive()

Mounted at /content/drive


In [3]:
mount_google_drive()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Configure Model Paths

In [ ]:
# Point these to your saved adapters on Google Drive
SFT_ADAPTER_PATH = "/content/drive/MyDrive/math-rl-tuning/sft"
SFT_MERGED_PATH  = "/content/drive/MyDrive/math-rl-tuning/sft_merged"   # merged SFT model (base for GRPO-from-SFT)
GRPO_FROM_SFT_PATH      = "/content/drive/MyDrive/math-rl-tuning/grpo_from_sft"
GRPO_FROM_INSTRUCT_PATH = "/content/drive/MyDrive/math-rl-tuning/grpo_from_instruct"

NUM_SAMPLES = 300  # number of eval examples per model

# Toggle: True = in-distribution val split (easier, ~higher numbers)
#         False = official NuminaMath test split
USE_VAL_SPLIT = True

## 3. Load Test Data

In [ ]:
from math_rl_tuning.data import prepare_test_data, prepare_eval_from_sft_sources

if USE_VAL_SPLIT:
    # In-distribution examples from SFT training sources (easier, held-out from training)
    test_ds = prepare_eval_from_sft_sources(cfg)
    print("Using in-distribution val split (same sources as SFT training)")
else:
    # Official NuminaMath-CoT test split
    test_ds = prepare_test_data(cfg)
    print("Using official test split")

print(f"Eval examples available: {len(test_ds)}")

## 4. Evaluate a Single Model

Use this to evaluate just one model (SFT or RL).

In [6]:
from math_rl_tuning.evaluation import evaluate_adapter

sft_df, sft_acc = evaluate_adapter(
    adapter_path=SFT_ADAPTER_PATH,
    test_dataset=test_ds,
    cfg=cfg,
    num_samples=NUM_SAMPLES,
    model_name="SFT",
)

print(f"\nSFT Accuracy: {sft_acc:.2%}")
sft_df.head(10)


Loading adapter: /content/drive/MyDrive/math-rl-tuning/sft
BnB compute dtype: torch.bfloat16
Loading base model: Qwen/Qwen2.5-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loading adapter: /content/drive/MyDrive/math-rl-tuning/sft
Evaluating SFT on 59 examples (batch_size=16)...


Batches:   0%|          | 0/4 [00:52<?, ?it/s]


KeyboardInterrupt: 

## 5. Five-Model Comparison

In [ ]:
from math_rl_tuning.evaluation import compare_five_models, save_results

results = compare_five_models(
    sft_adapter_path=SFT_ADAPTER_PATH,
    grpo_from_sft_adapter_path=GRPO_FROM_SFT_PATH,
    grpo_from_instruct_adapter_path=GRPO_FROM_INSTRUCT_PATH,
    sft_merged_path=SFT_MERGED_PATH,
    test_dataset=test_ds,
    cfg=cfg,
    num_samples=NUM_SAMPLES,
)

save_results(results, "./results/training_3.0_results")

## 6. Visualize Results

In [ ]:
import matplotlib.pyplot as plt

# Accuracy bar chart for all 5 models
model_labels = []
accuracies = []
colors = []

model_map = [
    ("base_accuracy",              "Base",                "#55A868"),
    ("instruct_accuracy",          "Instruct",            "#8172B2"),
    ("sft_accuracy",               "SFT",                 "#4C72B0"),
    ("grpo_from_sft_accuracy",     "GRPO-from-SFT",       "#DD8452"),
    ("grpo_from_instruct_accuracy","GRPO-from-Instruct",  "#C44E52"),
]

for key, label, color in model_map:
    if key in results:
        model_labels.append(label)
        accuracies.append(results[key])
        colors.append(color)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(model_labels, accuracies, color=colors, width=0.55)
ax.set_ylabel("Accuracy")
ax.set_title("Model Accuracy Comparison (5 Models)")
ax.set_ylim(0, max(accuracies) * 1.2 + 0.05)

for bar, acc in zip(bars, accuracies):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{acc:.1%}",
        ha="center",
        fontweight="bold",
    )

plt.tight_layout()
plt.savefig("accuracy_comparison_5models.png", dpi=150)
plt.show()

In [ ]:
def show_comparison(df, col_a, col_b, label_a, label_b):
    """Print cases where model B fixed model A's errors, and vice versa."""
    gains = df[(~df[f"{col_a}_correct"]) & df[f"{col_b}_correct"]]
    regressions = df[df[f"{col_a}_correct"] & (~df[f"{col_b}_correct"])]
    print(f"{label_b} fixed {len(gains)} {label_a} errors, regressed on {len(regressions)}.")
    if not gains.empty:
        print(f"\n=== {label_b} Improvements over {label_a} ===")
        display(gains[["problem_snippet", "ground_truth"]].reset_index(drop=True))

# Base → SFT
if "base_vs_sft" in results:
    print("── Base → SFT ──────────────────────────────")
    show_comparison(results["base_vs_sft"], "Base", "SFT", "Base", "SFT")

# SFT → RL
if "sft_vs_rl" in results:
    print("\n── SFT → RL ────────────────────────────────")
    show_comparison(results["sft_vs_rl"], "SFT", "RL", "SFT", "RL")

## 7. Cleanup

In [ ]:
from math_rl_tuning.utils import clean_memory
clean_memory()